In [ ]:
!rm ../new_project/llm_validation_checkpoint.parquet

# Targeted Outcome Validation with LLM

This utility allows you to boost the confidence of specific MedCAT annotations stored in your project database. 

**Workflow:**
1. Initialize the configuration used in the initial `pat2vec` run.
2. Fetch annotations for specific target concepts (e.g., specific CUIs or Pretty Names).
3. Use an LLM to confirm or reject these mentions based on the full clinical context.
4. Persist validation results back to the SQL database for downstream vectorization.

In [ ]:
import os
import pandas as pd
import requests
import logging
from datetime import datetime
from dateutil.relativedelta import relativedelta
from sqlalchemy import text, bindparam
from pat2vec.util.config_pat2vec import config_class
from med_llm_utils.medcat_llm_validator.validator import LLMAnnotationValidator
from dotenv import load_dotenv

# Setup logging for the notebook environment
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# --- 1. Replicate Pipeline Configuration ---
# Ensure these match the settings from your main example_usage.ipynb run
proj_name = 'new_project'
db_connection_string = "sqlite:///../new_project/outputs/temp_test_db_new_project.sqlite"
 # Use your actual connection string

config_obj = config_class(
    proj_name=proj_name,
    storage_backend="database",
    db_connection_string=db_connection_string,
    testing=True,
    testing_elastic=True,
    dummy_medcat_model=True,
    start_date=datetime(1995, 1, 1),
    years=30,
    time_window_interval_delta=relativedelta(years=31)
)


In [ ]:
from sqlalchemy import inspect

# 1. Identify existing annotation tables
engine = config_obj.db_engine
is_sqlite = engine.name == "sqlite"
inspector = inspect(engine)

# pat2vec uses the 'annotations' schema, which in SQLite is emulated as a prefix
schema_to_check = None if is_sqlite else "annotations"
existing_tables = inspector.get_table_names(schema=schema_to_check)

# Common annotation tables created by pat2vec
possible_tables = ['ann_epr_docs', 'ann_mct_docs', 'ann_textual_obs', 'ann_reports']

found_tables = []
for table in possible_tables:
    # Handle SQLite naming convention vs standard SQL schemas
    db_table_name = f"annotations_{table}" if is_sqlite else table
    if db_table_name in existing_tables:
        table_ref = f'"{db_table_name}"' if is_sqlite else f'"annotations"."{table}"'
        found_tables.append(table_ref)

if not found_tables:
    print("❌ No annotation tables found.")
    print(f"Tables currently in DB: {existing_tables}")
    print("Check if you have successfully run the MedCAT annotation step in your main pipeline.")
else:
    # 2. Build a query to aggregate all concepts across sources
    query_parts = [f"SELECT cui, pretty_name FROM {t}" for t in found_tables]
    combined_query = " UNION ALL ".join(query_parts)
    
    print(f"🔍 Inspecting concepts across: {', '.join(found_tables)}...")
    
    with engine.connect() as conn:
        df_all_found = pd.read_sql(text(combined_query), conn)

    # 3. Group and count to find the best candidates for validation
    summary = df_all_found.groupby(['cui', 'pretty_name']).size().reset_index(name='count')
    summary = summary.sort_values(by='count', ascending=False)

    print(f"✅ Found {len(summary)} unique concepts.")
    print("\nTop 20 candidates for validation (most frequent):")
    display(summary.head(20))
    
    # Optional: Save a list of CUIs for your 'target_concepts' list
    # print("\nCUI List for copy-pasting:")
    # print(summary['cui'].head(10).tolist())


### 2. Define Target Concepts and Local LLM Caller (Gemma)
We specify which codes (CUIs) we want to validate. Below is an example of how to connect to a local LLM like **Gemma 2b** running via Ollama.

In [ ]:
# Specify the list of codes or names to validate
target_concepts = ['267036007', 'Diabetes Mellitus', 'Diarrhea (finding)']


# 3. Enhanced Ollama Caller
def create_ollama_caller(model_name: str, host: str, timeout: int, is_reasoning: bool):
    def ollama_caller(prompt: str) -> str:
        url = f"http://{host}:11434/api/generate"
        options = {"temperature": 0.0, "stop": ["---END---"]}
        if is_reasoning:
            options.update({"num_ctx": 8192, "num_predict": 4096})

        payload = {"model": model_name, "prompt": prompt, "stream": False, "options": options}
        
        try:
            response = requests.post(url, json=payload, timeout=timeout)
            response.raise_for_status()
            data = response.json()
            
            # --- FIX: CAPTURE THE MONOLOGUE ---
            # Ollama reasoning models often put thinking in a separate 'reasoning' field
            resp_text = data.get("response", "")
            reasoning = data.get("reasoning", "")
            
            if reasoning:
                # Merge them so the validator sees the 'unstripped' raw output
                return f"<think>\n{reasoning}\n</think>\n{resp_text}"
            return resp_text
        except Exception as e:
            logger.error(f"Ollama call failed: {e}")
            return f"ERROR: {str(e)}"
            
    return ollama_caller

# --- CONFIGURATION ---
# Load variables from .env
load_dotenv()

# Access the variable
remote_server_ip = os.getenv("OLLAMA_REMOTE_IP", "127.0.0.1") # Fallback to localhost if not found
local_server_ip = os.getenv("OLLAMA_LOCAL_IP", "127.0.0.1")

# Access the variable
use_reasoning = True 
model = "qwen3:30b-thinking" if use_reasoning else "gemma2:9b"
host = remote_server_ip if use_reasoning else local_server_ip
print(f"Connecting to: {host}")
timeout = 300
my_caller = create_ollama_caller(model, host, timeout, use_reasoning)


In [ ]:
validator = LLMAnnotationValidator(
    db_engine=config_obj.db_engine,
    llm_caller=my_caller,
    text_column="text",
    is_reasoning_model=True,
    use_concise_prompts=False,
    dump_full_reasoning_response=True,
    checkpoint_path=f"../{proj_name}/llm_validation_checkpoint.parquet",
    checkpoint_interval=0,
    skip_availability_check=True,
    debug_mode=True
)


### 3. Retrieve Annotations from Database
We query the `ann_epr_docs` and `ann_mct_docs` tables created by the previous pat2vec run.

In [ ]:
from sqlalchemy import text, inspect
import pandas as pd

engine = config_obj.db_engine
is_sqlite = engine.name == "sqlite"
inspector = inspect(engine)

# In SQLite, pat2vec emulates schemas by prefixing table names
# In other DBs, we look inside the 'annotations' schema
schema_to_check = None if is_sqlite else "annotations"
existing_tables = inspector.get_table_names(schema=schema_to_check)

sources = {
    'epr': 'ann_epr_docs',
    'mct': 'ann_mct_docs',
    'textual_obs': 'ann_textual_obs'
}

query_parts = []
for source_id, table_name in sources.items():
    # Construct the actual table name used by pat2vec
    db_table_name = f"annotations_{table_name}" if is_sqlite else table_name
    
    if db_table_name in existing_tables:
        # Construct the SQL table reference
        table_ref = f'"{db_table_name}"' if is_sqlite else f'"annotations"."{table_name}"'
        
        query_parts.append(
            f"SELECT *, '{source_id}' as annotation_batch_source "
            f"FROM {table_ref} "
            f"WHERE cui IN :concepts OR pretty_name IN :concepts"
        )

if not query_parts:
    print("Warning: No annotation tables found in the database.")
    df_to_validate = pd.DataFrame()
else:
    raw_query = " UNION ALL ".join(query_parts)
    # Fix: Use expanding=True for the IN clause to work with list parameters
    stmt = text(raw_query).bindparams(bindparam("concepts", expanding=True))

    with engine.connect() as conn:
        df_to_validate = pd.read_sql(stmt, conn, params={"concepts": list(target_concepts)})

    print(f"Found {len(df_to_validate)} annotations matching the target criteria.")


In [ ]:
from sqlalchemy import text, inspect
import pandas as pd

# 1. Setup Database Inspection
engine = config_obj.db_engine
is_sqlite = engine.name == "sqlite"
inspector = inspect(engine)

# Get all tables to find the exact names
all_tables = inspector.get_table_names()

# 2. Identify potential matches for EPR or MCT data
# pat2vec uses these prefixes in SQLite
ann_candidates = [t for t in all_tables if "ann_epr" in t or "ann_mct" in t]
raw_candidates = [t for t in all_tables if "raw_epr" in t or "raw_mct" in t or "textual_obs" in t]

print(f"--- Database Inspection ({engine.name}) ---")
print(f"Annotation Tables Found: {ann_candidates}")
print(f"Raw Document Tables Found: {raw_candidates}")

def show_sample(table_name):
    print(f"\n>>> Table: {table_name}")
    with engine.connect() as conn:
        try:
            df = pd.read_sql(text(f'SELECT * FROM "{table_name}" LIMIT 1'), conn)
            if not df.empty:
                print(f"Columns: {df.columns.tolist()}")
                display(df)
            else:
                print("Result: Table is empty.")
        except Exception as e:
            print(f"Error reading table: {e}")

# 3. Display the samples
if ann_candidates:
    show_sample(ann_candidates[0])

if raw_candidates:
    # Try to show the raw table that matches the document type of the annotation table
    show_sample(raw_candidates[0])


In [ ]:
if not df_to_validate.empty:
    print(f"🚀 Starting validation for {len(df_to_validate)} annotations...")
    
    # Simple call: configuration is now handled internally by the validator instance
    validated_df = validator.process_dataframe(df_to_validate)
    
    # Persist results back to the database
    # Note: Ensure the validator has access to the database via db_engine
    validator._update_database_annotations(validated_df)
    
    print("✅ Validation complete. Database updated.")
    display(validated_df[['client_idcode', 'pretty_name', 'llm_status', 'llm_reason']].head())

In [ ]:
# Use this to view progress if the run above was interrupted
checkpoint_path = f"../{proj_name}/llm_validation_checkpoint.parquet"

if os.path.exists(checkpoint_path):
    progress_df = LLMAnnotationValidator.load_checkpoint_results(checkpoint_path)
    print(f"Current progress: {len(progress_df)} rows validated.")
    display(progress_df.head())
else:
    print("No checkpoint file found. Note: Check if the '!rm ...' cell at the top of the notebook deleted it.")


In [ ]:
progress_df['llm_status'].value_counts()

In [ ]:
validated_df